In [0]:
from pyspark.sql import SparkSession, functions as F 

  

spark = SparkSession.builder.appName("day3-assignment").getOrCreate() 

  

customers = (spark.read.option("header", True) 

             .schema("""customer_id INT, first_name STRING, last_name STRING, 

                        city STRING, state STRING, signup_date DATE""") 

             .csv("/Volumes/jaswanth/default/day-3_assignment/datasets/customers.csv")) 

  

orders = (spark.read.option("header", True) 

          .schema("""order_id INT, order_date DATE, customer_id INT, 

                     order_status STRING, amount DOUBLE""") 

          .csv("/Volumes/jaswanth/default/day-3_assignment/datasets/orders.csv")) 

  

products = (spark.read.option("header", True) 

            .schema("""product_id INT, product_name STRING, 

                       category STRING, price DOUBLE""") 

            .csv("/Volumes/jaswanth/default/day-3_assignment/datasets/products.csv")) 

In [0]:
customers.show(5)
products.display() 

+-----------+----------+---------+---------+-----+-----------+
|customer_id|first_name|last_name|     city|state|signup_date|
+-----------+----------+---------+---------+-----+-----------+
|          1|     Aarav|     Nair|Bengaluru|   KA| 2024-09-07|
|          2|    Vikram|    Reddy|   Mumbai|   MH| 2025-07-12|
|          3|     Varun|      Rao|Bengaluru|   KA| 2024-02-02|
|          4|     Rohan|    Patel|Hyderabad|   TG| 2024-08-26|
|          5|      Neha|   Sharma|   Mysuru|   KA| 2025-07-28|
+-----------+----------+---------+---------+-----+-----------+
only showing top 5 rows


product_id,product_name,category,price
1,Headphones,Electronics,4535.56
2,Keyboard,Electronics,4311.21
3,Mouse,Electronics,446.2
4,Monitor,Electronics,1265.22
5,Webcam,Electronics,3376.99
6,Python Basics,Books,1148.76
7,Spark Guide,Books,747.33
8,SQL Cookbook,Books,4683.02
9,Data Engineering 101,Books,2897.11
10,Desk Lamp,Home,2415.09


In [0]:
# Task 1.  Inspect a DataFrame 

customers.printSchema()
products.printSchema()
print(products.columns)
print(customers.columns)
print(customers.dtypes)
print(products.dtypes)
print(products.count())
customers.count()

root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)

['product_id', 'product_name', 'category', 'price']
['customer_id', 'first_name', 'last_name', 'city', 'state', 'signup_date']
[('customer_id', 'int'), ('first_name', 'string'), ('last_name', 'string'), ('city', 'string'), ('state', 'string'), ('signup_date', 'date')]
[('product_id', 'int'), ('product_name', 'string'), ('category', 'string'), ('price', 'double')]
20


50

In [0]:
# Task 2.  select and withColumn 
customers.show(5)
customers.select("customer_id","first_name").show(5)


+-----------+----------+---------+---------+-----+-----------+
|customer_id|first_name|last_name|     city|state|signup_date|
+-----------+----------+---------+---------+-----+-----------+
|          1|     Aarav|     Nair|Bengaluru|   KA| 2024-09-07|
|          2|    Vikram|    Reddy|   Mumbai|   MH| 2025-07-12|
|          3|     Varun|      Rao|Bengaluru|   KA| 2024-02-02|
|          4|     Rohan|    Patel|Hyderabad|   TG| 2024-08-26|
|          5|      Neha|   Sharma|   Mysuru|   KA| 2025-07-28|
+-----------+----------+---------+---------+-----+-----------+
only showing top 5 rows
+-----------+----------+
|customer_id|first_name|
+-----------+----------+
|          1|     Aarav|
|          2|    Vikram|
|          3|     Varun|
|          4|     Rohan|
|          5|      Neha|
+-----------+----------+
only showing top 5 rows


In [0]:
# Task 2.  select and withColumn 

full=customers.withColumn("full_name",F.concat_ws(" ","first_name","last_name"))
full.select("customer_id","full_name","city").show(5)

+-----------+------------+---------+
|customer_id|   full_name|     city|
+-----------+------------+---------+
|          1|  Aarav Nair|Bengaluru|
|          2|Vikram Reddy|   Mumbai|
|          3|   Varun Rao|Bengaluru|
|          4| Rohan Patel|Hyderabad|
|          5| Neha Sharma|   Mysuru|
+-----------+------------+---------+
only showing top 5 rows


In [0]:
# Task 3.  filter and orderBy

products.show(5)
(products.filter((F.col("category") == "Electronics") & (F.col("price") > 1000)) 
    .orderBy(F.col("price").desc()) 
    .select("product_name", "price") 
    .show(5))


+----------+------------+-----------+-------+
|product_id|product_name|   category|  price|
+----------+------------+-----------+-------+
|         1|  Headphones|Electronics|4535.56|
|         2|    Keyboard|Electronics|4311.21|
|         3|       Mouse|Electronics|  446.2|
|         4|     Monitor|Electronics|1265.22|
|         5|      Webcam|Electronics|3376.99|
+----------+------------+-----------+-------+
only showing top 5 rows
+------------+-------+
|product_name|  price|
+------------+-------+
|  Headphones|4535.56|
|    Keyboard|4311.21|
|      Webcam|3376.99|
|     Monitor|1265.22|
+------------+-------+



In [0]:
# Task 4.  distinct values 
display(customers.select(F.col("city")).distinct().orderBy(F.col("city").desc()))

city
Visakhapatnam
Pune
Mysuru
Mumbai
Kolkata
Hyderabad
Delhi
Coimbatore
Chennai
Bengaluru


In [0]:
# Task 5.  Your first Spark SQL query 

customers.createOrReplaceTempView("customers")
display(spark.sql("select * from customers"))

customer_id,first_name,last_name,city,state,signup_date
1,Aarav,Nair,Bengaluru,KA,2024-09-07
2,Vikram,Reddy,Mumbai,MH,2025-07-12
3,Varun,Rao,Bengaluru,KA,2024-02-02
4,Rohan,Patel,Hyderabad,TG,2024-08-26
5,Neha,Sharma,Mysuru,KA,2025-07-28
6,Riya,Rao,Mumbai,MH,2024-08-13
7,Varun,Nair,Visakhapatnam,AP,2024-01-07
8,Ishita,Gupta,Chennai,TN,2024-10-11
9,Arjun,Gupta,Chennai,TN,2024-04-14
10,Siddharth,Reddy,Bengaluru,KA,2025-01-02


In [0]:
# Task 5.  Your first Spark SQL query

customers.createOrReplaceTempView("customers")
display(spark.sql("""
    SELECT state, COUNT(*) AS customer_count
    FROM customers
    GROUP BY state
    ORDER BY customer_count DESC
"""))

state,customer_count
KA,14
MH,10
TN,10
DL,4
AP,4
TG,4
WB,4


In [0]:
# Task 6.  One question, two languages
# Dataframe API
print("DataFrame API")
api = (products
    .groupBy("category")
    .agg(F.count("product_id").alias("num_products"),
         F.round(F.avg("price"), 2).alias("avg_price"))
    .orderBy(F.col("avg_price").desc()))
api.show()

print("Spark SQL")
products.createOrReplaceTempView("products")
df_sql = spark.sql("""
    SELECT category,
           COUNT(product_id) AS num_products,
           ROUND(AVG(price), 2) AS avg_price
    FROM products
    GROUP BY category
    ORDER BY avg_price DESC
""")
df_sql.show()


DataFrame API
+-----------+------------+---------+
|   category|num_products|avg_price|
+-----------+------------+---------+
| Stationery|           3|  3997.12|
|       Home|           4|  2861.62|
|Electronics|           5|  2787.04|
|      Books|           4|  2369.06|
|    Fitness|           4|  1836.77|
+-----------+------------+---------+

Spark SQL
+-----------+------------+---------+
|   category|num_products|avg_price|
+-----------+------------+---------+
| Stationery|           3|  3997.12|
|       Home|           4|  2861.62|
|Electronics|           5|  2787.04|
|      Books|           4|  2369.06|
|    Fitness|           4|  1836.77|
+-----------+------------+---------+



In [0]:
# Task 7.  Join orders with customers

# Inner join on customer_id
joined = orders.join(customers, "customer_id", "inner")
print("Joined row count:", joined.count())
print("Orders total:", orders.count())
print("Customers total:", customers.count())

# Top 5 cities by number of orders
top_cities = (joined
    .groupBy("city")
    .count()
    .withColumnRenamed("count", "order_count")
    .orderBy(F.col("order_count").desc())
    .limit(5))
top_cities.show()

Joined row count: 294
Orders total: 300
Customers total: 50
+----------+-----------+
|      city|order_count|
+----------+-----------+
| Bengaluru|         49|
|    Mumbai|         41|
|   Chennai|         32|
|   Kolkata|         29|
|Coimbatore|         28|
+----------+-----------+

